In [49]:
import kagglehub
import os
import pandas as pd

In [50]:
# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Path to dataset files: /Users/hilal/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


In [51]:
# List all files in the dataset folder
files = os.listdir(path)

print("Files in dataset:")
for file in files:
    print(file)

Files in dataset:
olist_sellers_dataset.csv
product_category_name_translation.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv


In [ ]:
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

dfs = {}

for file in csv_files:
    name = file.replace(".csv", "")
    dfs[name] = pd.read_csv(os.path.join(path, file), dtype_backend='numpy_nullable', infer_datetime64_ns=True)

print(dfs.keys())

print(dfs["olist_orders_dataset"].dtypes)

dict_keys(['olist_sellers_dataset', 'product_category_name_translation', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset'])
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


In [ ]:
for name, df in dfs.items():
    print(f"\nDataset: {name}")
    print(df.columns.tolist())


Dataset: olist_sellers_dataset
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Dataset: product_category_name_translation
['product_category_name', 'product_category_name_english']

Dataset: olist_orders_dataset
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Dataset: olist_order_items_dataset
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Dataset: olist_customers_dataset
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Dataset: olist_geolocation_dataset
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Dataset: olist_order_payments_dataset
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Dataset: olist_o

In [ ]:
# Analyze duplicates in key columns for each dataset
print("=" * 70)
print("DUPLICATE ANALYSIS FOR EACH DATASET")
print("=" * 70)

# Define the key columns to check for duplicates in each dataset
key_columns = {
    "olist_orders_dataset": "order_id",
    "olist_customers_dataset": "customer_id",
    "olist_products_dataset": "product_id",
    "olist_order_reviews_dataset": "review_id",
    "olist_order_payments_dataset": "order_id",
    "olist_sellers_dataset": "seller_id",
    "olist_geolocation_dataset": "zip_code_prefix",
    "product_category_name_translation": "product_category_name"
}

duplicate_summary = {}

for dataset_name, df in dfs.items():
    print(f"\n📊 Dataset: {dataset_name}")
    print(f"   Shape: {df.shape}")
    
    # Get the key column for this dataset
    key_col = key_columns.get(dataset_name)
    
    if key_col and key_col in df.columns:
        # Count total rows and unique values
        total_rows = len(df)
        unique_count = df[key_col].nunique()
        duplicate_count = total_rows - unique_count
        
        # Find duplicate values
        duplicates = df[df.duplicated(subset=[key_col], keep=False)][key_col].value_counts()
        
        duplicate_summary[dataset_name] = {
            "key_column": key_col,
            "total_rows": total_rows,
            "unique_values": unique_count,
            "duplicate_rows": duplicate_count,
            "duplicate_percentage": (duplicate_count / total_rows * 100) if total_rows > 0 else 0
        }
        
        print(f"   Key Column: {key_col}")
        print(f"   Total Rows: {total_rows}")
        print(f"   Unique Values: {unique_count}")
        print(f"   Duplicate Rows: {duplicate_count} ({duplicate_summary[dataset_name]['duplicate_percentage']:.2f}%)")
        
        if duplicate_count > 0:
            print(f"   Top Duplicates in '{key_col}':")
            print(f"   {duplicates.head(5).to_dict()}")
    else:
        print(f"   ⚠️  Could not find key column '{key_col}' in dataset")
        print(f"   Available columns: {list(df.columns)}")

# Summary table
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
summary_df = pd.DataFrame(duplicate_summary).T
print(summary_df)

DUPLICATE ANALYSIS FOR EACH DATASET

📊 Dataset: olist_sellers_dataset
   Shape: (3095, 4)
   Key Column: seller_id
   Total Rows: 3095
   Unique Values: 3095
   Duplicate Rows: 0 (0.00%)

📊 Dataset: product_category_name_translation
   Shape: (71, 2)
   Key Column: product_category_name
   Total Rows: 71
   Unique Values: 71
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_orders_dataset
   Shape: (99441, 8)
   Key Column: order_id
   Total Rows: 99441
   Unique Values: 99441
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_order_items_dataset
   Shape: (112650, 7)
   ⚠️  Could not find key column 'None' in dataset
   Available columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

📊 Dataset: olist_customers_dataset
   Shape: (99441, 5)
   Key Column: customer_id
   Total Rows: 99441
   Unique Values: 99441
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_geolocation_dataset
   Shape: (1000163, 5)
   ⚠️  Could not find key column '